In [3]:
import pandas as pd
food = pd.read_csv("en.openfoodfacts.org.products.csv", sep= "\t", on_bad_lines= "skip") # load the csv file in while accounting for unique tab separation amd skipping corrupt lines

/var/folders/dh/wndstd9948b8myqxw6g0hlfh0000gn/T/ipykernel_32871/3950525275.py:2: DtypeWarning: Columns (0: code, 1: abbreviated_product_name, 2: packaging, 3: packaging_tags, 4: packaging_en, 5: packaging_text, 6: origins, 7: origins_tags, 8: origins_en, 9: manufacturing_places, 10: manufacturing_places_tags, 11: emb_codes, 12: emb_codes_tags, 13: first_packaging_code_geo, 14: cities, 15: cities_tags, 16: purchase_places, 17: stores, 18: allergens_en, 19: traces, 20: serving_quantity, 21: no_nutrition_data, 22: additives_n, 23: additives, 24: nutriscore_score, 25: brand_owner, 26: owner) have mixed types. Specify dtype option on import or set low_memory=False.
  food = pd.read_csv("en.openfoodfacts.org.products.csv", sep= "\t", on_bad_lines= "skip") # load the csv file in while accounting for unique tab separation amd skipping corrupt lines


In [29]:
food.head() # check to see if DataFrame was correctly created

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g
0,54,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1582569031,2020-02-24T18:30:31Z,1733085204,2024-12-01T20:33:24Z,NaN,1.740205e+09,2025-02-22T06:23:42Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,63,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1673620307,2023-01-13T14:31:47Z,1750061386,2025-06-16T08:09:46Z,bodysupport,1.750061e+09,2025-06-16T08:09:46Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,114,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1580066482,2020-01-26T19:21:22Z,1751035658,2025-06-27T14:47:38Z,teolemon,1.751036e+09,2025-06-27T14:47:38Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,431,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1714301712,2024-04-28T10:55:12Z,1714301721,2024-04-28T10:55:21Z,kiliweb,1.714302e+09,2024-04-28T10:55:21Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,105,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1572117743,2019-10-26T19:22:23Z,1738073570,2025-01-28T14:12:50Z,NaN,1.743653e+09,2025-04-03T04:11:36Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
food.dtypes # check which types of data is present and if anything must be converted

code                         object
url                             str
creator                         str
created_t                     int64
created_datetime                str
                             ...   
sulphate_100g               float64
nitrate_100g                float64
acidity_100g                float64
carbohydrates-total_100g    float64
water_100g                  float64
Length: 211, dtype: object

In [31]:
product_category = food[["product_name", "brands", "brands_tags"]] # create a subset with only the relevant columns for brand name extraction

In [32]:
product_category.head() # check to make sure subset was correctly created

,product_name,brands,brands_tags
0,Limonade artisanale a la rose,NaN,NaN
1,M&amp;M white,Fitpiggy,xx:fitpiggy
2,Chocolate n3,Jeff de Bruges,xx:jeff-de-bruges
3,Pâte de fruits,NaN,NaN
4,Paleta gran reserva - Sierra nevada-,AdvoCare,xx:advocare


In [33]:
complete_products = product_category.dropna(subset= "product_name") # drop missing product rows as they provide no value to our purpose of extracting brand names from product titles

In [34]:
complete_products.head() # check to make sure no "NaNs" are present

,product_name,brands,brands_tags
0,Limonade artisanale a la rose,NaN,NaN
1,M&amp;M white,Fitpiggy,xx:fitpiggy
2,Chocolate n3,Jeff de Bruges,xx:jeff-de-bruges
3,Pâte de fruits,NaN,NaN
4,Paleta gran reserva - Sierra nevada-,AdvoCare,xx:advocare


In [35]:
complete_products.count() # check to find missing data

product_name    4196933
brands          2774494
brands_tags     2774973
dtype: int64

In [36]:
import html
import re
import unicodedata

def normalize(text):
    """Lowercase, decode HTML entities, strip accents, and collapse whitespace so that
    brand/title comparisons are insensitive to casing, encoding artifacts, and accent variants."""
    text = html.unescape(text) # decode HTML entities such as &amp; and &quot;
    text = text.lower()
    text = re.sub(r"[\u2018\u2019\u02bc\u0060\u00b4]", "'", text) # fold typographic/curly apostrophe variants to a straight ' so e.g. "welch's" and "welch's" compare equal (done before NFKD, which would otherwise turn the acute accent into a space)
    text = unicodedata.normalize("NFKD", text) # split accented characters into base character + combining mark
    text = "".join(char for char in text if not unicodedata.combining(char)) # drop the combining marks, leaving just the base characters
    return re.sub(r"\s+", " ", text).strip() # collapse any run of whitespace to a single space and trim the ends

normalized_products = complete_products.copy() # copy so we don't mutate complete_products in place
normalized_products["product_name_norm"] = normalized_products["product_name"].apply(normalize) # normalized titles for matching; raw product_name kept intact for readability and the future LLM approach

In [37]:
normalized_products.head() # check to make sure normalization was successful

,product_name,brands,brands_tags,product_name_norm
0,Limonade artisanale a la rose,NaN,NaN,limonade artisanale a la rose
1,M&amp;M white,Fitpiggy,xx:fitpiggy,m&m white
2,Chocolate n3,Jeff de Bruges,xx:jeff-de-bruges,chocolate n3
3,Pâte de fruits,NaN,NaN,pate de fruits
4,Paleta gran reserva - Sierra nevada-,AdvoCare,xx:advocare,paleta gran reserva - sierra nevada-


In [38]:
dupes = normalized_products[normalized_products.duplicated()] # check to see if there are duplicated rows
dupes

,product_name,brands,brands_tags,product_name_norm
352,Nutritional yeast,NaN,NaN,nutritional yeast
495,Chips de banane,NaN,NaN,chips de banane
517,Nutritional yeast,NaN,NaN,nutritional yeast
543,Erbsen,NaN,NaN,erbsen
653,Nutritional yeast,NaN,NaN,nutritional yeast
...,...,...,...,...
4532721,Steak haché,NaN,NaN,steak hache
4532744,جاري التحميل…,NaN,NaN,جاري التحميل...
4532746,Steak haché 5%,NaN,NaN,steak hache 5%
4532751,Marrons glacés,NaN,NaN,marrons glaces


In [39]:
# deduplication now happens once, after all normalization (see the drop_duplicates on the normalized columns further below), so case/accent/spacing variants are also caught

In [40]:
brands_known = normalized_products.dropna(subset= ["brands"]) # we need the brand field present to establish ground truth, so drop empty brand field rows
brands_known = brands_known.dropna(subset= "product_name") # redundant, just in case
brands_known["brands_norm"] = brands_known["brands"].apply(normalize) # normalized brands for matching; raw brands kept intact

In [41]:
brands_known.head() # quick check to make sure no missing brand rows present, at least in a small sample

,product_name,brands,brands_tags,product_name_norm,brands_norm
1,M&amp;M white,Fitpiggy,xx:fitpiggy,m&m white,fitpiggy
2,Chocolate n3,Jeff de Bruges,xx:jeff-de-bruges,chocolate n3,jeff de bruges
4,Paleta gran reserva - Sierra nevada-,AdvoCare,xx:advocare,paleta gran reserva - sierra nevada-,advocare
12,granola Bio le Chocolaté,Mg ricarica,xx:mg-ricarica,granola bio le chocolate,mg ricarica
14,xytitol pastilles,xylimgxyling,xx:xylimgxyling,xytitol pastilles,xylimgxyling


In [42]:
print(brands_known['brands'].apply(type).value_counts()) # make sure all brand entries are strings
print(brands_known['product_name'].apply(type).value_counts()) # make sure all product name entries are strings

brands
<class 'str'>    2774494
Name: count, dtype: int64
product_name
<class 'str'>    2774494
Name: count, dtype: int64


In [43]:
import difflib

# generic-word / ingredient lists used to reject spurious fuzzy matches (e.g. title "chocolate" -> brand "chocolatier")
with open("1-1000.txt", encoding= "utf-8") as f:
    common_words_set = {normalize(line) for line in f if line.strip()}
food_ingredients_set = set()
food_ingredient_tokens_set = set()
with open("10000_common_food_ingredients.txt", encoding= "utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        phrase = normalize(line)
        food_ingredients_set.add(phrase)
        food_ingredient_tokens_set.update(phrase.split())

def fuzzy_threshold_for_brand(brand):
    """Minimum SequenceMatcher ratio for a near match; shorter brands require a higher threshold."""
    length = len(brand.replace(" ", "")) # character length, ignoring spaces
    if length <= 3:
        return 0.97
    if length <= 5:
        return 0.93
    if length <= 8:
        return 0.92
    if length <= 12:
        return 0.85
    return 0.82

def is_generic_match_window(window):
    """True when the title text being fuzzy-matched is a common English word or food ingredient/product."""
    if window in common_words_set or window in food_ingredients_set:
        return True
    if len(window.split()) == 1 and window in food_ingredient_tokens_set:
        return True
    return False

def _tokenize_title(product):
    """Split a normalized title on whitespace and strip attached commas per token."""
    tokens = []
    for piece in product.split():
        token = piece.strip(",")
        if token:
            tokens.append(token)
    return tokens

def _token_char_offset(tokens, index):
    if index == 0:
        return 0
    return len(" ".join(tokens[:index])) + 1

def first_brand_in_title(product, brands):
    """The brands field can hold multiple comma-separated brands; return the one that appears
    earliest in the product name as a whole word, or None if none of them appear.
    Ties at the same position go to the longer brand, e.g. "tesco finest" over "tesco"."""
    tokens = _tokenize_title(product)
    matches = []
    for brand in (b.strip() for b in brands.split(",")):
        if not brand:
            continue
        brand_tokens = brand.split()
        width = len(brand_tokens)
        for i in range(len(tokens) - width + 1):
            if tokens[i : i + width] == brand_tokens:
                matches.append((_token_char_offset(tokens, i), -len(brand), brand))
                break
    return min(matches)[2] if matches else None

def near_brand_in_title(product, brands):
    """Return the brand with the earliest near match in the product name, or None if there is none.
    A near match is a same-token-count window of the title that is fuzzily similar to the brand,
    catching misspellings/transliterations such as brand "jagermeister" in title "jaegermeister liqueur".
    Rejects windows that are generic English words or known food ingredients/products."""
    tokens = _tokenize_title(product)
    best = None # (window index, -brand length, brand) so min() picks the earliest position, longest brand on ties
    for brand in (b.strip() for b in brands.split(",")):
        if not brand:
            continue
        threshold = fuzzy_threshold_for_brand(brand)
        width = len(brand.split())
        for i in range(len(tokens) - width + 1):
            window = " ".join(tokens[i : i + width])
            if abs(len(window) - len(brand)) > 3 or window[0] != brand[0]:
                continue # cheap pre-filters: a near match can't differ much in length and should share its first letter
            if is_generic_match_window(window):
                continue # ingredient/common-word title token, not a real brand occurrence
            if difflib.SequenceMatcher(None, brand, window).ratio() >= threshold:
                candidate = (i, -len(brand), brand)
                best = candidate if best is None else min(best, candidate)
                break # earliest window for this brand found, no need to scan further right
    return best[2] if best else None

def categorize(product, brands):
    """Segment an entry by how recoverable its brand is from the (normalized) title:
    category 1 = brand appears verbatim, 2 = brand inferable via near match, 3 = brand absent.
    Also returns the brand that was found (None for category 3)."""
    if (brand := first_brand_in_title(product, brands)) is not None:
        return 1, brand
    if (brand := near_brand_in_title(product, brands)) is not None:
        return 2, brand
    return 3, None

brands_known_products = brands_known.drop_duplicates(subset= ["product_name_norm", "brands_norm"]) # single deduplication pass on the normalized columns so case/accent/spacing variants collapse
categories = [categorize(product, brands) for product, brands in zip(brands_known_products["product_name_norm"], brands_known_products["brands_norm"])]
brands_known_products = brands_known_products.assign(
    match_category= [category for category, _ in categories], # 1 = exact, 2 = inferable, 3 = absent
    brands_matched= [brand for _, brand in categories], # the (normalized) brand recovered from the title, None for category 3
    gold_brand= brands_known_products["brands"].apply(normalize), # ground-truth ("gold") label: the raw brands field run through normalize()
)
has_ground_truth = brands_known_products[brands_known_products["match_category"] < 3] # products whose brand is recoverable from the title, exactly or via inference

In [44]:
print(brands_known_products["match_category"].value_counts().sort_index()) # how many products fall into each category
print()
print(brands_known_products["match_category"].value_counts(normalize= True).sort_index()) # and the same breakdown as proportions

match_category
1     241648
2       9061
3    2130378
Name: count, dtype: int64

match_category
1    0.101486
2    0.003805
3    0.894708
Name: proportion, dtype: float64


In [45]:
brands_known_products[brands_known_products["match_category"] == 2][["product_name_norm", "brands_norm", "brands_matched"]].head(20) # eyeball near matches to sanity-check the fuzzy threshold

,product_name_norm,brands_norm,brands_matched
197,vegan brain + body boost,"rum dunkel, vegan brain food",vegan brain food
393,datu putti spicy adobo,datu puti,datu puti
445,villa de varda (trie grappa) pinot nero,grappa,grappa
562,hershey s milk nuggets,hershey's,hershey's
1233,canola harvest® original vegetable oil spread tub,canola harvest,canola harvest
1608,teegschwender gut's nachtle,teegschwendner,teegschwendner
1889,herbalife 24,haerfbalife,haerfbalife
2174,roter heringssalat 1 kg schale,"heringsalat, salate im kubel",heringsalat
2221,"kartoffelsalat mit speck 5,00 kg eimer",kartoffesalat,kartoffesalat
2269,pringle,pringles,pringles


In [46]:
try:
    sample = pd.read_csv("benchmark.csv", index_col= 0) # reuse the frozen benchmark so the rows stay stable across re-runs
    loaded_from_disk = True
except FileNotFoundError:
    # first run only: draw a fresh stratified benchmark (the next cell saves it to benchmark.csv)
    GROUND_TRUTH_SAMPLE_SIZE = 500 # categories 1 & 2: brand recoverable from the title, the core extraction test
    BRAND_ABSENT_SAMPLE_SIZE = 250 # category 3: brand absent from the title, held out to test behavior when there is nothing to extract
    ground_truth_sample = has_ground_truth.sample(n= GROUND_TRUTH_SAMPLE_SIZE, random_state= 10) # same seed as before, so these rows match the previous benchmark exactly
    brand_absent_sample = brands_known_products[brands_known_products["match_category"] == 3].sample(n= BRAND_ABSENT_SAMPLE_SIZE, random_state= 10)
    sample = pd.concat([ground_truth_sample, brand_absent_sample]) # stratified benchmark: brand-present rows plus a segment-3 block
    loaded_from_disk = False

# frozen rows keep stable indices, but segment labels must follow the current categorize() logic
if loaded_from_disk:
    sample[["match_category", "brands_matched"]] = brands_known_products.loc[
        sample.index, ["match_category", "brands_matched"]
    ]

# dedicated segment-2 (fuzzy/inferable) stratum: category 2 is only ~0.4% of the data, so the
# proportional ground-truth draw captured too few (~8) to estimate fuzzy accuracy reliably. Top up
# to SEG2_TARGET by drawing additional category-2 rows not already sampled. This only ever ADDS rows
# (never drops), so the existing frozen benchmark stays stable across re-runs.
SEG2_TARGET = 150
seg2_pool = brands_known_products[brands_known_products["match_category"] == 2]
current_seg2 = sample.index[sample["match_category"] == 2]
seg2_needed = SEG2_TARGET - len(current_seg2)
if seg2_needed > 0:
    extra_seg2 = seg2_pool.drop(index=sample.index, errors="ignore").sample(
        n=seg2_needed, random_state=10
    )
    sample = pd.concat([sample, extra_seg2[sample.columns]])

# manual re-segmentation: fuzzy matching mapped an ingredient/common product word onto a brand for these
# entries (e.g. "aloe"->"alo", "quark"->"quarki", "chocolate"->"chocolatier", "quinoa"->"quinola",
# "muesli"->"mymuesli"), so they were mislabeled category 2; they are truly brand-absent (segment 3). Force
# them every run, regardless of whether the benchmark was loaded from disk or freshly drawn.
FORCE_ABSENT_INDICES = [1017265, 2576829, 4461157, 2236122, 2582746]
force_absent = sample.index.isin(FORCE_ABSENT_INDICES)
sample.loc[force_absent, "match_category"] = 3 # segment 3: brand absent from the title
sample.loc[force_absent, "brands_matched"] = None # the fuzzy "match" was spurious, so no brand was actually recovered

In [47]:
sample

,product_name,brands,brands_tags,product_name_norm,brands_norm,match_category,brands_matched
1017265,"Real Dried Aloe Vera + Ginger Snacks, Ginger",ALO,xx:alo,"real dried aloe vera + ginger snacks, ginger",alo,3,NaN
3007397,"Wawa, Chicken Steak Burrito Bowl",Wawa,wawa,"wawa, chicken steak burrito bowl",wawa,1,wawa
29773,"Long Grove Confectionery Co., St. Pat's Mini P...",Long Grove Confectionery Co.,long-grove-confectionery-co,"long grove confectionery co., st. pat's mini p...",long grove confectionery co.,1,long grove confectionery co.
1255714,"PF Chang's, Tempura Calamari & Vegetables",PF Chang's,pf-chang-s,"pf chang's, tempura calamari & vegetables",pf chang's,1,pf chang's
3151484,Terra Delyssa Ekstra jomfruolivenolie øko,Terra Delyssa,xx:terra-delyssa,terra delyssa ekstra jomfruolivenolie øko,terra delyssa,1,terra delyssa
...,...,...,...,...,...,...,...
3978789,Ensalada completa césar pollo y queso,Florette,xx:florette,ensalada completa cesar pollo y queso,florette,3,NaN
775455,Almond Salted Caramel Bars,Honey Bar,honey-bar,almond salted caramel bars,honey bar,3,NaN
4524253,Beurre aux cristaux de sel de Guérande,La motte,xx:la-motte,beurre aux cristaux de sel de guerande,la motte,3,NaN
3623017,Missanga de chocolate drage,Oba,xx:oba,missanga de chocolate drage,oba,3,NaN


In [48]:
sample.to_csv("benchmark.csv", index= True) # save the benchmark sample; keep the index so each row traces back to its original position in food

In [49]:
benchmark_100 = sample.sample(n= 100, random_state= 10) # random 100-row subset of the benchmark sample
with pd.option_context("display.max_rows", 100): # show all 100 rows just for this cell, without changing the global display setting
    display(benchmark_100)

,product_name,brands,brands_tags,product_name_norm,brands_norm,match_category,brands_matched
4212074,AH Servetten 2-laags 33x33 cm wit,AH,xx:albert-heijn,ah servetten 2-laags 33x33 cm wit,ah,1,ah
4128378,"Quiznos, Dressing, For Classic Italian, 8&quot;",Quiznos,quiznos,"quiznos, dressing, for classic italian, 8""",quiznos,1,quiznos
58603,Mountain Dew Froth Cherry-Lemon,Mountain Dew,xx:mountain-dew,mountain dew froth cherry-lemon,mountain dew,1,mountain dew
2531116,Ice cream,Aldi,aldi,ice cream,aldi,3,NaN
1398674,Oat Bran - Kazidomi,Kazidomi,xx:kazidomi,oat bran - kazidomi,kazidomi,1,kazidomi
2642423,Chester,Landgut,xx:landgut,chester,landgut,3,NaN
2582746,Mozart muesli,mymuesli,xx:mymuesli,mozart muesli,mymuesli,3,NaN
2690421,Активиа Биойогурт груша черника,Danone,danone,активиа биоиогурт груша черника,danone,3,NaN
2236122,Quinoa mais tomates haricot rouge,Quinola,xx:quinola,quinoa mais tomates haricot rouge,quinola,3,NaN
1574385,Exotic Party,Palais des Thés,xx:palais-des-thes,exotic party,palais des thes,3,NaN


In [50]:
print(f"Proportion of products with brand recoverable from title (categories 1 & 2): {len(has_ground_truth)/len(food)}")
print(f"Proportion of products with brand absent from title (category 3): {(brands_known_products['match_category'] == 3).sum()/len(food)}")

Proportion of products with brand recoverable from title (categories 1 & 2): 0.05531039001580713
Proportion of products with brand absent from title (category 3): 0.46999524572749746


In [51]:
# common_words_set and food ingredient sets are loaded earlier (with fuzzy matching) for spurious-match rejection
print(f"loaded {len(common_words_set)} common words; {len(food_ingredients_set)} food ingredient phrases; first 10 common words: {sorted(common_words_set)[:10]}")

loaded 1000 common words; 10000 food ingredient phrases; first 10 common words: ['a', 'able', 'about', 'above', 'act', 'add', 'afraid', 'after', 'again', 'against']


In [52]:
# curated entries that demonstrate each edge-case type, hand-picked from the full deduped set (brands_known_products) so they resolve regardless of which rows land in the sample
vague_examples = brands_known_products.loc[[2903202, 391053, 3513687]].to_dict("records") # vague: These are entries where the brand and/or the product names are a common word in the English language/ far too vague and could be matched incorrectly (especially for the LLM). Keep in mind, some of these entries will not be marked by the program as vague since the criteria for that is based on the top 1000 most common words in the English language appearing as brand names, that is to say, while these entries could be classified as vague in such a small sample, in the grand scheme of things they are not as vague.
unique_chars = brands_known_products.loc[[2808808, 4329388, 2275905, 3200240, 4530373, 3681026]].to_dict("records") # special chars: non-English text, other alphabets, or special characters that are uncommon and are worth checking to see if extraction breaks in either implementation.
misspelled_examples = brands_known_products.loc[[1668569]].to_dict("records") # misspelled: the brand name is misspelled in the product title, for this reason we have to ensure fuzzy matching does not break down here (e.g. brand "apericube" appears as "aperocube")
brand_conflict_examples = brands_known_products.loc[[3487928, 1691422, 2653847]].to_dict("records") # brand_conflict: a multi-brand entry where all brands appear in the title, so the correct brand is ambiguous to extract
spurious_substring_examples = brands_known_products.loc[[4351314]].to_dict("records") # spurious_substring: an unassociated brand appears inside the title as a substring (e.g. brand "u" inside "uncle chipps"). This example is a very sensitive instance, in reality, we would like to have a coincidental substring of a longer length.
noisy_examples = brands_known_products.loc[[2653847]].to_dict("records") # noisy: an exceptionally long product title packed with extra descriptors/codes, making brand extraction harder
example_groups = {
    "vague": vague_examples,
    "unique_chars": unique_chars,
    "misspelled": misspelled_examples,
    "brand_conflict": brand_conflict_examples,
    "spurious_substring": spurious_substring_examples,
    "noisy": noisy_examples,
} # each curated list demonstrates one edge-case type
examples_display = pd.concat([pd.DataFrame(entries).assign(edge_case= label) for label, entries in example_groups.items()], ignore_index= True) # tag each example with the edge case it demonstrates
examples_display[["product_name", "brands", "brands_matched", "edge_case"]] # display the demonstrative edge-case entries alongside their edge-case type

,product_name,brands,brands_matched,edge_case
0,Galaxy Chocolate milk,Galaxy,galaxy,vague
1,Polar Seltzer Black Cherry,Polar,polar,vague
2,Sweet Chilli Philadelphia,Philadelphia,philadelphia,vague
3,Co-op fusilli,Co-op,co-op,unique_chars
4,Est play Kamikaze,"เอส,Est",est,unique_chars
5,Вуйчо Ваньо,Вуйчо Ваньо,вуичо ваньо,unique_chars
6,nescafe brasero crema,Nescafé,nescafe,unique_chars
7,"Auntie Anne's, Gold Peak, Peach Tea, 21 Oz",Auntie Anne's,auntie anne's,unique_chars
8,Pomodorina Menu (830g Dose),Menù,menu,unique_chars
9,Aperocube,Apericube,apericube,misspelled


In [53]:
# brand universe: every distinct brand seen anywhere in the dataset, used to spot brands that appear spuriously inside unrelated titles
MIN_SPURIOUS_BRAND_LEN = 3 # minimum brand length to consider; raise to ignore noise from very short brands like "u"
brand_universe = {brand.strip() for brands in brands_known_products["brands_norm"] for brand in str(brands).split(",") if len(brand.strip()) >= MIN_SPURIOUS_BRAND_LEN}
max_brand_len = max(len(brand) for brand in brand_universe) # cap substring scanning to the longest brand

NOISY_LENGTH_THRESHOLD = 60 # product titles longer than this (about the 95th percentile / mean + 2*std of title lengths) are exceptionally long and noisy

def is_vague(brands_norm):
    """A brand that is itself a common English word (top-1000 list) or a food ingredient/product name
    is too generic and risks being matched incorrectly."""
    for brand in (b.strip() for b in str(brands_norm).split(",") if b.strip()):
        if brand in common_words_set or brand in food_ingredients_set:
            return True
        if len(brand.split()) == 1 and brand in food_ingredient_tokens_set:
            return True
    return False

def has_non_latin_text(text):
    """True when text contains a genuine non-Latin script or other non-ASCII that survives accent normalization.
    Accented Latin (e.g. é, ñ) folds to ASCII and is not flagged; plain punctuation is benign."""
    return any(ord(char) > 127 for char in normalize(str(text)))

def has_unique_chars(product_name, brands):
    """Flag non-Latin scripts or non-ASCII text after accent normalization in either field, or a space inside
    an individual brand name. Plain punctuation (dashes, apostrophes, commas, etc.) is benign."""
    if has_non_latin_text(product_name) or has_non_latin_text(brands):
        return True
    return any(" " in brand.strip() for brand in str(brands).split(",")) # spaces count as special only inside a brand name

def has_brand_conflict(product_name_norm, brands_norm):
    """A multi-brand entry where every listed brand appears in the title as a whole word, so the correct
    brand is ambiguous to extract (e.g. the actual brand vs the retailer, especially if it is not listed first)."""
    brands = [brand.strip() for brand in str(brands_norm).split(",") if brand.strip()]
    if len(brands) < 2:
        return False # needs at least two brands to conflict
    return all(re.search(rf"\b{re.escape(brand)}\b", str(product_name_norm)) for brand in brands) # every brand must appear in the title

def has_spurious_substring(product_name_norm, brands_norm):
    """Flag only entries where a naive (no word-boundary) substring matcher would extract the WRONG
    brand: the longest brand from the dataset's brand universe that occurs anywhere in the title is an
    unrelated brand sitting embedded inside a larger word (so word-boundary matching skips it), while
    none of the entry's own brands appear cleanly as a standalone token. These are exactly the cases
    where dropping word boundaries breaks extraction (e.g. brand "u" inside "uncle chipps")."""
    product = str(product_name_norm)
    own = {brand.strip() for brand in str(brands_norm).split(",") if brand.strip()}
    if any(re.search(rf"\b{re.escape(brand)}\b", product) for brand in own):
        return False # a correct brand is cleanly present, so the right answer is still recoverable
    naive_match = None # the brand the naive matcher would pick: longest universe substring in the title
    best_len = 0
    n = len(product)
    for i in range(n):
        for j in range(min(n, i + max_brand_len), i, -1): # try the longest substring at i first
            piece = product[i:j]
            if piece in brand_universe:
                if j - i > best_len:
                    naive_match, best_len = piece, j - i # strict > keeps the earliest on length ties
                break # longest match starting at i found; shorter ones here cannot win
    if naive_match is None or naive_match in own:
        return False # nothing matched, or the naive match is actually one of the entry's own brands
    return re.search(rf"\b{re.escape(naive_match)}\b", product) is None # spurious only if embedded

def has_noisy(product_name):
    """An exceptionally long product title carries lots of extra descriptors/codes/numbers, making brand extraction noisy."""
    return len(str(product_name)) > NOISY_LENGTH_THRESHOLD

def edge_case_label(row):
    """List every edge-case type an entry demonstrates (comma-joined), or None for an ordinary entry."""
    labels = []
    if is_vague(row["brands_norm"]):
        labels.append("vague")
    if row["match_category"] == 2:
        labels.append("misspelled") # category 2 means the brand was recoverable only via fuzzy near-match, i.e. it is misspelled/variant in the title
    if has_brand_conflict(row["product_name_norm"], row["brands_norm"]):
        labels.append("brand_conflict")
    if has_spurious_substring(row["product_name_norm"], row["brands_norm"]):
        labels.append("spurious_substring")
    if has_unique_chars(row["product_name"], row["brands"]):
        labels.append("unique_chars")
    if has_noisy(row["product_name"]):
        labels.append("noisy")
    return ", ".join(labels) if labels else None

sample["edge_case"] = sample.apply(edge_case_label, axis= 1) # label each entry by the edge case(s) it demonstrates; None for ordinary entries
print(sample["edge_case"].value_counts(dropna= False).to_string()) # how many entries fall into each edge case
sample[sample["edge_case"].notna()][["product_name", "brands", "brands_matched", "edge_case"]] # display entries demonstrating edge cases, note that NaN means a non-edge-case entry

edge_case
NaN                                    411
unique_chars                           240
spurious_substring                      33
spurious_substring, unique_chars        28
unique_chars, noisy                     13
noisy                                   10
misspelled                               5
vague                                    2
brand_conflict, unique_chars             2
misspelled, unique_chars                 2
vague, unique_chars                      1
misspelled, unique_chars, noisy          1
brand_conflict                           1
brand_conflict, unique_chars, noisy      1


,product_name,brands,brands_matched,edge_case
29773,"Long Grove Confectionery Co., St. Pat's Mini P...",Long Grove Confectionery Co.,long grove confectionery co.,unique_chars
1255714,"PF Chang's, Tempura Calamari & Vegetables",PF Chang's,pf chang's,unique_chars
3151484,Terra Delyssa Ekstra jomfruolivenolie øko,Terra Delyssa,terra delyssa,unique_chars
136511,"Giant eagle, thin swiss cheese, a natural firm...","Giant Eagle, Giant Eagle Inc.",giant eagle,"unique_chars, noisy"
426209,Hello Kitty Strawberry Chewy Candy,Hello Kitty,hello kitty,unique_chars
...,...,...,...,...
36562,Sparkling Mineral Water,Private Selection,NaN,unique_chars
3781496,Dop Terra di Bari “Castel del Monte”,Ecor,NaN,unique_chars
775455,Almond Salted Caramel Bars,Honey Bar,NaN,unique_chars
4524253,Beurre aux cristaux de sel de Guérande,La motte,NaN,unique_chars
